In [ ]:
# info


# OPTIMIZED
# run_results = joblib.load("../results/run_2024-01-16_18:42:07_results.job")
# run_model = joblib.load("../results/run_2024-01-16_18:42:07_model.job")

# NON OPTIMIZED

# run_results = joblib.load("../results/run_2024-01-17_11:07:58_results.job")
# run_model = joblib.load("../results/run_2024-01-17_11:07:58_model.job")

# det optimized

# run_results = joblib.load("../results/run_2024-01-22_20:29:42_results.job")
# run_model = joblib.load("../results/run_2024-01-22_20:29:42_model.job")


# true_parameters = game.true_pars

# number_of_samples = 500

# iters = np.arange(0, 4000)

In [1]:
import numpy as np
import jax
import jax.numpy as jnp

jax.config.update("jax_platform_name", "cpu")
from jax import vmap, jit
import matplotlib.pyplot as plt
import joblib
from qdots_qll.models import game
from qdots_qll import all_funcs
import seaborn as sns
import pandas as pd
import scipy
from functools import reduce
import os
import re
from scipy.stats import binned_statistic

import seaborn as sns


import matplotlib.font_manager as font_manager

import equinox as eqx

# from matplotlib import rcParams
from scipy.stats import binned_statistic


font = {"family": "Inter"}  # , 'weight': 'normal', 'size': 12}

# Aplica la fuente definida a Matplotlib
plt.rc("font", **font)

sns.set_palette("colorblind")

In [2]:
names_true = [
    "$\\gamma ( - \\eta)$",
    "$\\gamma ( + \\eta)$",
    "$S ( - \\eta)$",
    "$S ( +\\eta)$",
]
names_hat = [
    "$\\hat{\\gamma} ( - \\eta)$",
    "$\\hat{\\gamma} ( + \\eta)$",
    "$\\hat{S} ( - \\eta)$",
    "$\\hat{S} ( +\\eta)$",
]

In [3]:
def compute_mean_one_run(cum_times_i, cov_arr_i, limites_bins):
    indices_bins = np.digitize(cum_times_i, limites_bins)
    means = np.array(
        [
            np.nanmean(cov_arr_i[indices_bins == i], axis=0)
            for i in range(1, len(limites_bins))
        ]
    )

    std_devs = np.array(
        [
            np.nanstd(cov_arr_i[indices_bins == i], axis=0)
            for i in range(1, len(limites_bins))
        ]
    )
    return means, std_devs


def get_binned_results_from_runs(
    times_array,
    cov_array,
    est_array,
    bin_step,
):
    dims_cov = cov_array[:, 1:].shape[2:]
    dims_est = est_array[:, 1:].shape[2:]

    cum_times = np.array(times_array[:, 1:]).cumsum(axis=1)
    cum_times_flatten = cum_times.flatten()
    cov_flatten = np.array(cov_array[:, 1:]).reshape(-1, *dims_cov)
    est_flatten = np.array(est_array[:, 1:]).reshape(-1, *dims_est)

    bins = np.arange(
        cum_times_flatten.min(), cum_times_flatten.max() + 1, bin_step
    )

    cov_mean_list = []
    cov_std_list = []

    for i in range(dims_cov[0]):
        cov_mean_row_list = []
        cov_std_row_list = []

        for j in range(dims_cov[1]):

            cov_mean_row_list.append(
                binned_statistic(
                    cum_times_flatten,
                    cov_flatten[:, i, j],
                    statistic="mean",
                    bins=bins,
                )[0]
            )
            cov_std_row_list.append(
                binned_statistic(
                    cum_times_flatten,
                    cov_flatten[:, i, j],
                    statistic="std",
                    bins=bins,
                )[0]
            )
        cov_mean_list.append(cov_mean_row_list)
        cov_std_list.append(cov_std_row_list)
    cov_mean_list = np.array(cov_mean_list).transpose(2, 1, 0)
    cov_std_list = np.array(cov_std_list).transpose(2, 1, 0)

    est_mean_list = []
    est_std_list = []

    for i in range(dims_est[0]):

        est_mean_list.append(
            binned_statistic(
                cum_times_flatten,
                est_flatten[
                    :,
                    i,
                ],
                statistic="mean",
                bins=bins,
            )[0]
        )
        est_std_list.append(
            binned_statistic(
                cum_times_flatten,
                est_flatten[
                    :,
                    i,
                ],
                statistic="std",
                bins=bins,
            )[0]
        )
    est_mean_list = np.array(est_mean_list).transpose(1, 0)
    est_std_list = np.array(est_std_list).transpose(1, 0)

    cum_times_binned, _, _ = binned_statistic(
        cum_times_flatten, cum_times_flatten, statistic="mean", bins=bins
    )
    return (
        cum_times_binned,
        cov_mean_list,
        cov_std_list,
        est_mean_list,
        est_std_list,
    )

In [4]:
def results_bin_single_run(times, covs, ests, no_bins):

    cum_times = np.array(times[1:]).cumsum()
    # bins = np.linspace(cum_times.min(), cum_times.max() + 1, no_bins)
    dims_cov = covs.shape[-2:]
    dims_est = ests.shape[-1:]

    cov_mean_list = []
    cov_std_list = []

    for i in range(dims_cov[0]):
        cov_mean_row_list = []
        cov_std_row_list = []

        for j in range(dims_cov[1]):

            cov_mean_row_list.append(
                binned_statistic(
                    cum_times,
                    covs[1:, i, j],
                    statistic="mean",
                    bins=no_bins,
                )[0]
            )
            cov_std_row_list.append(
                binned_statistic(
                    cum_times,
                    covs[1:, i, j],
                    statistic="std",
                    bins=no_bins,
                )[0]
            )
        cov_mean_list.append(cov_mean_row_list)
        cov_std_list.append(cov_std_row_list)
    cov_mean_list = np.array(cov_mean_list).transpose(2, 1, 0)
    cov_std_list = np.array(cov_std_list).transpose(2, 1, 0)

    est_mean_list = []
    est_std_list = []

    for i in range(dims_est[0]):

        est_mean_list.append(
            binned_statistic(
                cum_times,
                ests[
                    1:,
                    i,
                ],
                statistic="mean",
                bins=no_bins,
            )[0]
        )
        est_std_list.append(
            binned_statistic(
                cum_times,
                ests[
                    1:,
                    i,
                ],
                statistic="std",
                bins=no_bins,
            )[0]
        )
    est_mean_list = np.array(est_mean_list).transpose(1, 0)
    est_std_list = np.array(est_std_list).transpose(1, 0)

    cum_times_binned, _, _ = binned_statistic(
        cum_times, cum_times, statistic="mean", bins=no_bins
    )
    times_binned, _, _ = binned_statistic(
        cum_times, np.array(times[1:]), statistic="mean", bins=no_bins
    )
    return (
        times_binned,
        cum_times_binned,
        cov_mean_list,
        cov_std_list,
        est_mean_list,
        est_std_list,
    )

In [ ]:
from qdots_qll.models import game


model = joblib.load("../results/run_2024-01-22_20:29:42_model.job")
true_parameters = game.true_pars
number_of_samples = 500

In [ ]:
filenames = sorted(os.listdir("../results_cluster/one_qdot/"))
filenames = ["../results_cluster/one_qdot/" + i for i in filenames]
job_filenames = list(filter(re.compile(".*job").match, filenames))
log_filenames = list(filter(re.compile(".*log").match, filenames))

runs = [joblib.load(i) for i in job_filenames]